# Regressão Logística do Zero com PyTorch

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.



Neste tutorial vamos implementar regressão logística do zero com PyTorch. O modelo será projetado pensando em redes neurais e usado em uma tarefa simples de classificação de imagens. É uma ótima abordagem para entender os blocos fundamentais por trás de uma rede neural. Além disso, vemos boas práticas de uso do PyTorch no treinamento.

Ao terminar, espera-se que o leitor conheça os blocos básicos de um modelo de regressão logística e consiga aplicá-lo a um problema próprio de classificação binária com PyTorch.

---


In [ ]:
## Import the usual libraries
import torch
import torchvision
import torch.nn as nn
from torchvision import datasets, models, transforms
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
## configuration to detect cuda or cpu
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print (device)

## Importando o Conjunto de Dados

Vamos trabalhar em um problema de classificação de imagens. O conjunto público está [aqui](https://download.pytorch.org/tutorial/hymenoptera_data.zip).

O objetivo do modelo é aprender a classificar entre "bee" (abelha) e "no bee" (sem abelha).

Descomente o código abaixo para baixar e descompactar os dados.


In [ ]:
# download the data
!wget https://download.pytorch.org/tutorial/hymenoptera_data.zip
!unzip hymenoptera_data.zip

## Transformação dos Dados

É uma tarefa de classificação de imagens, então precisamos aplicar algumas transformações antes de treinar. Para detalhes de cada transformação, veja a [documentação oficial do torchvision](https://pytorch.org/docs/stable/torchvision/transforms.html).

O bloco abaixo faz:

- `data_transforms` contém uma série de transformações aplicadas a cada imagem: crop, resize, conversão para tensor, reshape e normalização.
- Definidas as transformações, `DataLoader` carrega o conjunto e configura shuffle, batches etc.


In [ ]:
# configure root folder on your gdrive
data_dir = 'hymenoptera_data'

# custom transformer to flatten the image tensors
class ReshapeTransform:
    def __init__(self, new_size):
        self.new_size = new_size

    def __call__(self, img):
        result = torch.reshape(img, self.new_size)
        return result

# transformations used to standardize and normalize the datasets
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        ReshapeTransform((-1,)) # flattens the data
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        ReshapeTransform((-1,)) # flattens the data
    ]),
}

# load the correspoding folders
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}

# load the entire dataset; we are not using minibatches here
train_dataset = torch.utils.data.DataLoader(image_datasets['train'],
                                            batch_size=len(image_datasets['train']),
                                            shuffle=True)

test_dataset = torch.utils.data.DataLoader(image_datasets['val'],
                                           batch_size=len(image_datasets['val']),
                                           shuffle=True)

In [ ]:
len(image_datasets['train']), len(image_datasets['val'])

## Imprimir Amostra

Sempre é boa prática dar uma olhada no conjunto antes de treinar. Abaixo imprimimos um exemplo de imagem do `train_dataset`.


In [ ]:
# load the entire dataset
x, y = next(iter(train_dataset))

# print one example
dim = x.shape[1]
print("Dimension of image:", x.shape, "\n", 
      "Dimension of labels", y.shape)

plt.imshow(x[160].reshape(1, 3, 224, 224).squeeze().T.numpy())

## Construindo o Modelo

Vamos implementar nosso modelo de [regressão logística](https://en.wikipedia.org/wiki/Logistic_regression). Regressão logística é uma técnica de aprendizado de máquina usada para treinar classificadores binários. Também é uma ótima forma de entender os blocos fundamentais de uma rede neural — pode ser vista como a rede neural mais simples, em que o modelo realiza `forward` e `backward` para treinar com os dados.

Em resumo:

- `__init__` inicializa todos os parâmetros (`W`, `b`, `grad`) usados no treinamento via retropropagação.
- O objetivo é aprender `W` e `b` que minimizem a função de custo calculada na função `loss` abaixo.

Note que esta é uma implementação bem detalhada, então movi explicitamente várias computações para GPU via `to(device)`.


In [ ]:
class LR(nn.Module):
    def __init__(self, dim, lr=torch.scalar_tensor(0.01)):
        super(LR, self).__init__()
        # intialize parameters
        self.w = torch.zeros(dim, 1, dtype=torch.float).to(device)
        self.b = torch.scalar_tensor(0).to(device)
        self.grads = {"dw": torch.zeros(dim, 1, dtype=torch.float).to(device),
                      "db": torch.scalar_tensor(0).to(device)}
        self.lr = lr.to(device)

    def forward(self, x):
        # compute forward
        z = torch.mm(self.w.T, x) + self.b
        a = self.sigmoid(z)
        return a

    def sigmoid(self, z):
        # compute sigmoid
        return 1/(1 + torch.exp(-z))

    def backward(self, x, yhat, y):
        # compute backward
        self.grads["dw"] = (1/x.shape[1]) * torch.mm(x, (yhat - y).T)
        self.grads["db"] = (1/x.shape[1]) * torch.sum(yhat - y)
    
    def optimize(self):
        # optimization step
        self.w = self.w - self.lr * self.grads["dw"]
        self.b = self.b - self.lr * self.grads["db"]

## utility functions
def loss(yhat, y):
    m = y.size()[1]
    return -(1/m)* torch.sum(y*torch.log(yhat) + (1 - y)* torch.log(1-yhat))

def predict(yhat, y):
    y_prediction = torch.zeros(1, y.size()[1])
    for i in range(yhat.size()[1]):
        if yhat[0, i] <= 0.5:
            y_prediction[0, i] = 0
        else:
            y_prediction[0, i] = 1
    return 100 - torch.mean(torch.abs(y_prediction - y)) * 100

## Pré-teste do Modelo

Também é boa prática testar o modelo e garantir que os passos certos acontecem antes de iniciar o treino completo.


In [ ]:
# model pretesting
x, y = next(iter(train_dataset))

# flatten/transform the data
x_flatten = x.T
y = y.unsqueeze(0) 

# num_px is the dimension of the images
dim = x_flatten.shape[0]

# model instance
model = LR(dim)
model.to(device)
yhat = model.forward(x_flatten.to(device))
yhat = yhat.data.cpu()

# calculate loss
cost = loss(yhat, y)
prediction = predict(yhat, y)
print("Cost: ", cost)
print("Accuracy: ", prediction)

# backpropagate
model.backward(x_flatten.to(device), yhat.to(device), y.to(device))
model.optimize()

## Treinar o Modelo

Hora de treinar.


In [ ]:
# hyperparams
costs = []
dim = x_flatten.shape[0]
learning_rate = torch.scalar_tensor(0.0001).to(device)
num_iterations = 500
lrmodel = LR(dim, learning_rate)
lrmodel.to(device)

# transform the data
def transform_data(x, y):
    x_flatten = x.T
    y = y.unsqueeze(0) 
    return x_flatten, y 

# train the model
for i in range(num_iterations):
    x, y = next(iter(train_dataset))
    test_x, test_y = next(iter(test_dataset))
    x, y = transform_data(x, y)
    test_x, test_y = transform_data(test_x, test_y)

    # forward
    yhat = lrmodel.forward(x.to(device))
    cost = loss(yhat.data.cpu(), y)
    train_pred = predict(yhat, y)
        
    # backward
    lrmodel.backward(x.to(device), 
                    yhat.to(device), 
                    y.to(device))
    lrmodel.optimize()

    # test
    yhat_test = lrmodel.forward(test_x.to(device))
    test_pred = predict(yhat_test, test_y)

    if i % 10 == 0:
        costs.append(cost)

    if i % 10 == 0:
        print("Cost after iteration {}: {} | Train Acc: {} | Test Acc: {}".format(i, 
                                                                                    cost, 
                                                                                    train_pred,
                                                                                    test_pred))

## Resultado

Pela curva de perda abaixo dá pra ver que o modelo está aprendendo a classificar as imagens (perda decrescente). Rodei apenas `100` iterações. Treine por muito mais rodadas e analise os resultados. Sugiro alguns experimentos no fim do tutorial.


In [ ]:
## the trend in the context of loss
plt.plot(costs)
plt.show()

## Notas

Vários melhorias e experimentos possíveis:

- Sempre é bom normalizar/padronizar as imagens — ajuda no aprendizado. Experimente diferentes formas de padronização.
- Mexer no learning rate ajuda muito. Tente reduzir e aumentar e observe o efeito.
- Se explorar o conjunto, vai notar que todas as imagens "no-bee" são na verdade formigas. Para um modelo mais robusto, deixe os negativos mais diversos via data augmentation.
- O modelo não vai muito bem com regressão logística simples. Pode ser pelo conjunto e pelo treino curto. Tente em outros conjuntos.
- Falta uma análise completa dos resultados. Crie um conjunto pequeno de teste para validar a generalização.
- Construímos do zero, mas com PyTorch dá para usar módulos prontos. Como exercício, reescreva uma versão mais concisa.


## Referências

- [Understanding Impact of Learning Rate on Neural Network Performance](https://machinelearningmastery.com/understand-the-dynamics-of-learning-rate-on-deep-learning-neural-networks/)
- [Transfer Learning for Computer Vision Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
